In [1]:
!wget https://data.ncl.ac.uk/ndownloader/articles/24574678/versions/1 -O ../data/MultNIST.zip
!mkdir ../data/MultNIST
!unzip ../data/MultNIST.zip -d ../data/MultNIST/
!rm ../data/MultNIST.zip

--2025-05-09 22:42:01--  https://data.ncl.ac.uk/ndownloader/articles/24574678/versions/1
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 52.49.76.148, 52.48.143.184, 54.171.30.6
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|52.49.76.148|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 659123404 (629M) [application/zip]
Saving to: ‘../data/MultNIST.zip’

../data/MultNIST.zi 100%[===================>] 628.59M  33.1MB/s    in 23s     

2025-05-09 22:42:24 (27.7 MB/s) - ‘../data/MultNIST.zip’ saved [659123404/659123404]

Archive:  ../data/MultNIST.zip
 extracting: ../data/MultNIST/metadata  
 extracting: ../data/MultNIST/test_y.npy  
 extracting: ../data/MultNIST/train_y.npy  
 extracting: ../data/MultNIST/valid_y.npy  
 extracting: ../data/MultNIST/test_x.npy  
 extracting: ../data/MultNIST/valid_x.npy  
 extracting: ../data/MultNIST/train_x.npy  
 extracting: ../data/MultNIST/README  


In [1]:
!nvidia-smi

Thu May 15 14:21:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.120                Driver Version: 550.120        CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA TITAN RTX               Off |   00000000:42:00.0 Off |                  N/A |
| 57%   66C    P8             22W /  280W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [3]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=3 * 28 * 28, output_size=10):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [4]:
hyperparams = {
    "num_epochs": 128,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.3},
    "pruning_thresholds": {"fc0": 0.1},
    "plateau_threshold": 0.005,
    "min_epochs_between_expansions": 12,
    "plateau_window_size": 10,
    "learning_rate": 2e-4,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 5000,
    "weight_decay": 5e-3
}

In [5]:
train_x = np.load("../data/MultNIST/train_x.npy")
train_y = np.load("../data/MultNIST/train_y.npy")
valid_x = np.load("../data/MultNIST/valid_x.npy")
valid_y = np.load("../data/MultNIST/valid_y.npy")
test_x = np.load('../data/MultNIST/test_x.npy')
test_y = np.load('../data/MultNIST/test_y.npy')


train_tensor_x = torch.tensor(train_x).float().view(-1, 3 * 28 * 28)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 3 * 28 * 28)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 3 * 28 * 28)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

In [6]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [ ]:
import wandb
wandb.login()
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-MultNIST",
    name=f"trash",
    config=hyperparams
)

In [8]:
import time
start_time = time.time()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, val_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:01<00:00, 115.45it/s]


Epoch 1/128, Train Loss: 1.9315, Val Loss: 1.7385, Val Accuracy: 0.2959


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:01<00:00, 147.11it/s]


Epoch 2/128, Train Loss: 1.7336, Val Loss: 1.6663, Val Accuracy: 0.3086


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:01<00:00, 143.90it/s]


Epoch 3/128, Train Loss: 1.6882, Val Loss: 1.6376, Val Accuracy: 0.3083


... (246 outputs left out)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:11<00:00, 17.68it/s]


Epoch 127/128, Train Loss: 0.4302, Val Loss: 0.8792, Val Accuracy: 0.6876
Chosen edges: tensor([[   0,    0,    0,  ...,    9,    2,    8],
        [ 234,  297,  325,  ..., 9738, 9739, 9739]], device='cuda:0') 578


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:11<00:00, 16.73it/s]


Epoch 128/128, Train Loss: 0.4311, Val Loss: 0.8600, Val Accuracy: 0.6955
--- 2565.403641462326 seconds ---


In [9]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [10]:
accuracy, params

(0.6028, 5008164)

In [ ]:
torch.save(sparse_model, 'multnist.pt')

In [11]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 91.55, 94.01, 90.43, 64.20, 87.70, 92.81, 90.57, 96.68, 98.10, 97.17, 96.55, 39.76, 10],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))

styled_table

Model,Accuracy (%),Parameters
Ours,60.28,"5,008,164.0"
ResNet-18,91.55,"11,689,512.0"
AlexNet,94.01,"61,100,840.0"
VGG16,90.43,"138,357,544.0"
ConvNext,64.20,"88,591,464.0"
MNASNet,87.70,"4,383,312.0"
DenseNet,92.81,"28,681,000.0"
ResNeXt,90.57,"25,028,904.0"
PC-DARTS,96.68,nan
DrNAS,98.10,nan
